# 🔍 Notebook 2: Hybrid RAG Search (BM25 + Dense FAISS)
**EduNavika** -- Combining Exact Keyword Matching with Semantic Vector Search.

In educational applications, students search for exact terms (e.g. `Ohm's Law`, `Section formula`) as well as conceptual questions (e.g. `How does electricity flow?`).
EduNavika uses **Reciprocal Rank Fusion (RRF)** to combine both!


### Step 1: Lexical BM25 Search
BM25 handles exact keyword searches, acronyms, and formulas.


In [ ]:
from backend.app.retrieval.lexical.bm25 import BM25Index

sample_chunks = [
    {'id': 'c1', 'text': 'Real numbers include rational and irrational numbers. The fundamental theorem of arithmetic applies to composites.'},
    {'id': 'c2', 'text': 'Ohm\'s law states that electric current is directly proportional to voltage across a conductor: V = IR.'},
    {'id': 'c3', 'text': 'Pythagoras theorem states that in a right angled triangle, hypotenuse squared equals sum of squares of other two sides.'}
]

bm25 = BM25Index()
bm25.build(sample_chunks)

query = 'What is Ohm\'s law and voltage?'
results = bm25.search(query, top_k=2)
for r in results:
    print(f'Chunk: {r.chunk_id} | Score: {r.score:.3f}')


### Step 2: Dense Semantic Embeddings
Dense vectors capture meaning and concept similarities even when words are different.


In [ ]:
from backend.app.retrieval.embeddings.sentence_transformer import SentenceTransformerEmbedding

embedder = SentenceTransformerEmbedding('all-MiniLM-L6-v2')
query_vector = embedder.embed_text('How does electric potential relate to current?')
print(f'Embedding Vector Dimension: {len(query_vector)}')
print(f'Vector Preview (first 5 dimensions): {query_vector[:5]}')


### Step 3: Reciprocal Rank Fusion (RRF)
RRF merges BM25 ranking and Vector ranking into a single unified score:


In [ ]:
from backend.app.retrieval.rrf import reciprocal_rank_fusion

lexical_rankings = [{'chunk_id': 'c2', 'score': 10.5}, {'chunk_id': 'c1', 'score': 4.2}]
dense_rankings = [{'chunk_id': 'c2', 'score': 0.88}, {'chunk_id': 'c3', 'score': 0.65}]

hybrid = reciprocal_rank_fusion([lexical_rankings, dense_rankings], k=60)
print('Top Hybrid Results:')
for h in hybrid:
    print(f'Chunk {h["chunk_id"]} -> RRF Score: {h["rrf_score"]:.4f}')
